# Drosophila Brain Simulation - GPU Accelerated

138,639 LIF neurons from FlyWire v783 on free Colab T4 GPU.

Expected: ~90x faster than CPU (1.2ms/step vs 90ms/step)

## 1. Setup

In [0]:
!nvidia-smi
import torch
print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available(): print(f"GPU: {torch.cuda.get_device_name(0)}")

In [0]:
!git clone https://github.com/pusulamkendim/flywire-neuro.git
%cd flywire-neuro
!pip install -q fastapi uvicorn websockets pydantic pandas pyarrow pyngrok nest_asyncio

## 2. Download data from Zenodo

In [0]:
import os, urllib.request
DATA_DIR = "fly-brain-embodied/data"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs("data", exist_ok=True)
COMP_FILE = f"{DATA_DIR}/2025_Completeness_783.csv"
CONN_FILE = f"{DATA_DIR}/2025_Connectivity_783.parquet"
ANN_FILE = "data/neuron_annotations.tsv"
files = {
    COMP_FILE: "https://zenodo.org/records/10676866/files/2025_Completeness_783.csv",
    CONN_FILE: "https://zenodo.org/records/10676866/files/2025_Connectivity_783.parquet",
    ANN_FILE: "https://raw.githubusercontent.com/flyconnectome/flywire_annotations/main/supplemental_files/Supplemental_file1_neuron_annotations.tsv",
}
for fp, url in files.items():
    if os.path.exists(fp): print(f"OK {fp}")
    else:
        print(f"Downloading {os.path.basename(fp)}...")
        urllib.request.urlretrieve(url, fp)
        print(f"  Done ({os.path.getsize(fp)//1024//1024}MB)")
print("All data ready!")

## 3. Load brain on GPU

In [0]:
import sys, time
import torch, numpy as np, pandas as pd
sys.path.insert(0, "fly-brain-embodied/code")
sys.path.insert(0, "fly-brain-embodied")
import benchmark
benchmark.COMP_PATH = COMP_FILE
benchmark.CONN_PATH = CONN_FILE
benchmark.DATA_DIR = DATA_DIR
from run_pytorch import TorchModel, MODEL_PARAMS, DT, get_weights, get_hash_tables
from brain_body_bridge import DN_NEURONS, DN_GROUPS, STIMULI, DNRateDecoder
flyid2i, i2flyid = get_hash_tables(COMP_FILE)
num_neurons = len(flyid2i)
device = "cuda" if torch.cuda.is_available() else "cpu"
t0 = time.time()
weights = get_weights(CONN_FILE, COMP_FILE, DATA_DIR).to(device=device)
print(f"Neurons: {num_neurons:,} | Device: {device} | Loaded in {time.time()-t0:.1f}s")

## 4. Benchmark

In [0]:
model = TorchModel(batch=1, size=num_neurons, dt=DT, params=MODEL_PARAMS, weights=weights, device=device)
state = model.state_init()
rates = torch.zeros(1, num_neurons, device=device)
for _ in range(5):
    with torch.no_grad(): state = model(rates, *state)
if device=="cuda": torch.cuda.synchronize()
t0 = time.time()
for _ in range(100):
    with torch.no_grad(): state = model(rates, *state)
if device=="cuda": torch.cuda.synchronize()
ms = (time.time()-t0)/100*1000
print(f"Speed: {ms:.2f} ms/step | {1.0/(ms*10/1000):.0f} fps | {90/ms:.0f}x vs CPU")

## 5. Setup populations + server

In [0]:
ann = pd.read_csv("data/neuron_annotations.tsv", sep="\t", low_memory=False)
def ids_to_tensor(root_ids):
    return torch.tensor([flyid2i[nid] for nid in root_ids if nid in flyid2i], dtype=torch.long, device=device)
mbon = ann[ann["cell_class"] == "MBON"]
ser = ann[(ann["top_nt"] == "serotonin") & (~ann["cell_class"].isin(["olfactory","visual","mechanosensory","unknown_sensory","gustatory"]))]
octo = ann[(ann["top_nt"] == "octopamine") & (~ann["cell_class"].isin(["olfactory","visual","mechanosensory","unknown_sensory","gustatory"]))]
pop_tensors = {
    "pam": ids_to_tensor(ann[ann["cell_type"].str.startswith("PAM", na=False)]["root_id"]),
    "ppl1": ids_to_tensor(ann[ann["cell_type"].str.startswith("PPL1", na=False)]["root_id"]),
    "mbon_approach": ids_to_tensor(mbon[mbon["top_nt"] == "acetylcholine"]["root_id"]),
    "mbon_avoidance": ids_to_tensor(mbon[mbon["top_nt"] == "glutamate"]["root_id"]),
    "mbon_suppress": ids_to_tensor(mbon[mbon["top_nt"] == "gaba"]["root_id"]),
    "serotonin": ids_to_tensor(ser["root_id"]),
    "octopamine": ids_to_tensor(octo["root_id"]),
    "gaba": ids_to_tensor(ann[ann["top_nt"] == "gaba"]["root_id"]),
    "ach": ids_to_tensor(ann[ann["top_nt"] == "acetylcholine"]["root_id"]),
    "glut": ids_to_tensor(ann[ann["top_nt"] == "glutamate"]["root_id"]),
}
stim_map = {sn: [flyid2i[nid] for nid in si["neurons"] if nid in flyid2i] for sn, si in STIMULI.items()}
dn_map = {name: flyid2i[fid] for name, fid in DN_NEURONS.items() if fid in flyid2i}
print(f"Populations: {len(pop_tensors)} | DN: {len(dn_map)} | Stimuli: {list(stim_map.keys())}")

In [0]:
import asyncio, json, threading
from fastapi import FastAPI, WebSocket
from fastapi.responses import HTMLResponse
import uvicorn

app = FastAPI()
active_stimuli = set()
brain_lock = threading.Lock()

# Write HTML to file to avoid string escaping issues
html_content = """<html><head><title>Fly Brain GPU</title>
<style>
body{background:#0a0a12;color:#e0e0f0;font-family:monospace;padding:20px}
.btn{padding:8px 16px;margin:4px;border:2px solid #444;border-radius:6px;background:#1a1a28;color:#aaa;cursor:pointer;font-size:14px}
.btn.active{color:white;border-color:#2ecc71;background:rgba(46,204,113,0.2)}
.br{display:flex;align-items:center;margin:3px 0}
.bl{width:130px;font-size:12px;color:#888}
.bt{flex:1;height:14px;background:#111;border-radius:3px;overflow:hidden}
.bf{height:100%;border-radius:3px;transition:width 0.05s}
.bv{width:60px;font-size:11px;color:#666;text-align:right}
h2{color:#556;font-size:13px;text-transform:uppercase;letter-spacing:2px;margin-top:20px}
</style></head><body>
<h1>Drosophila Brain - GPU</h1>
<p style="color:#556">138,639 LIF neurons | FlyWire v783 | CUDA</p>
<div id="st" style="color:#2ecc71">Connecting...</div>
<div id="info" style="font-size:11px;color:#555"></div>
<h2>Stimulus Control</h2><div id="sb"></div>
<h2>Descending Neurons</h2><div id="dn"></div>
<h2>Dopamine</h2><div id="da"></div>
<h2>Neuromodulators</h2><div id="nm"></div>
<h2>Bulk NT</h2><div id="nt"></div>
<script>
var S=[["p9","P9 Walk","#9b59b6"],["lc4","LC4 Loom","#e74c3c"],["sugar","Sugar","#ff9800"],["bitter","Bitter","#c0392b"],["jo","JO Touch","#00bcd4"],["or56a","Or56a","#27ae60"]];
var B={dn:[["escape","GF Escape","#e74c3c"],["forward","P9 Forward","#2ecc71"],["backward","MDN Back","#9b59b6"],["turn_L","Turn L","#3498db"],["turn_R","Turn R","#e67e22"],["groom","aDN1 Groom","#00bcd4"],["feed","MN9 Feed","#ff9800"]],da:[["pam","PAM","#2ecc71"],["ppl1","PPL1","#e74c3c"],["mbon_approach","MBON App","#27ae60"],["mbon_avoidance","MBON Avd","#c0392b"]],nm:[["serotonin","5-HT","#9b59b6"],["octopamine","OA","#e67e22"]],nt:[["gaba","GABA","#3498db"],["ach","ACh","#27ae60"],["glut","Glut","#e84393"]]};
var act=new Set(),ws,mx={},fc=0,lf=performance.now(),fps=0;
function bb(id,d){var c=document.getElementById(id);d.forEach(function(b){c.innerHTML+='<div class="br"><span class="bl">'+b[1]+'</span><div class="bt"><div class="bf" id="b-'+b[0]+'" style="width:0%;background:'+b[2]+'"></div></div><span class="bv" id="v-'+b[0]+'">0</span></div>'})}
function bs(){var c=document.getElementById("sb");S.forEach(function(s){var b=document.createElement("button");b.className="btn";b.textContent=s[1];b.onclick=function(){b.classList.toggle("active");if(b.classList.contains("active")){act.add(s[0]);b.style.borderColor=s[2]}else{act.delete(s[0]);b.style.borderColor=""}if(ws&&ws.readyState===1)ws.send(JSON.stringify({cmd:"set_stimuli",stimuli:Array.from(act)}))};c.appendChild(b)})}
function cn(){var p=location.protocol==="https:"?"wss:":"ws:";ws=new WebSocket(p+"//"+location.host+"/ws");ws.onopen=function(){document.getElementById("st").textContent="Connected - GPU"};ws.onclose=function(){document.getElementById("st").textContent="Disconnected";setTimeout(cn,2000)};ws.onmessage=function(e){var d=JSON.parse(e.data);if(!d.dn)return;B.dn.forEach(function(b){var v=d.dn[b[0]]||0;document.getElementById("b-"+b[0]).style.width=Math.min(100,v*100)+"%";document.getElementById("v-"+b[0]).textContent=v.toFixed(3)});["da","nm","nt"].forEach(function(g){B[g].forEach(function(b){var v=d.pop[b[0]]||0;mx[b[0]]=Math.max(mx[b[0]]||1,v,1);document.getElementById("b-"+b[0]).style.width=Math.min(100,v/mx[b[0]]*100)+"%";document.getElementById("v-"+b[0]).textContent=v})});fc++;var n=performance.now();if(n-lf>=1000){fps=fc;fc=0;lf=n}document.getElementById("info").textContent="t="+d.t_ms.toFixed(0)+"ms | "+d.behavior+" | "+d.total_spikes+" spikes | "+fps+" fps"}}
bs();bb("dn",B.dn);bb("da",B.da);bb("nm",B.nm);bb("nt",B.nt);cn();
</script></body></html>"""

@app.get("/")
async def root():
    return HTMLResponse(html_content)

@app.websocket("/ws")
async def ws_ep(websocket: WebSocket):
    global active_stimuli
    await websocket.accept()
    decoder = DNRateDecoder(window_ms=50.0, dt_ms=DT, max_rate=200.0)
    r = torch.zeros(1, num_neurons, device=device)
    st = model.state_init()
    sa = torch.zeros(num_neurons, device=device)
    ps = set()
    step = 0
    async def rx():
        global active_stimuli
        try:
            while True:
                msg = await websocket.receive_text()
                c = json.loads(msg)
                if c.get("cmd") == "set_stimuli":
                    with brain_lock: active_stimuli = set(c.get("stimuli", []))
        except: pass
    async def sim():
        nonlocal r, st, sa, ps, step
        try:
            while True:
                with brain_lock: cur = active_stimuli.copy()
                if cur != ps:
                    r.zero_()
                    for sn in cur:
                        if sn in stim_map:
                            for idx in stim_map[sn]: r[0, idx] = STIMULI[sn]["rate"]
                    ps = cur.copy()
                with torch.no_grad(): st = model(r, *st)
                sa += st[2][0]
                decoder.update({n: st[2][0, i].item() for n, i in dn_map.items()})
                step += 1
                if step % 10 == 0:
                    dn = {g: round(decoder.get_group_rate(g), 4) for g in DN_GROUPS}
                    gf = dn.get("escape", 0)
                    beh = "escape" if gf>0.06 else "grooming" if dn.get("groom",0)>0.02 else "feeding" if dn.get("feed",0)>0.05 else "walking" if dn.get("forward",0)>0.01 else "idle"
                    pop = {n: int(sa[i].sum().item()) for n, i in pop_tensors.items()}
                    tot = int(sa.sum().item()); sa.zero_()
                    try: await websocket.send_json({"t_ms": round(step*DT,1), "step": step, "dn": dn, "pop": pop, "total_spikes": tot, "behavior": beh})
                    except: break
                    await asyncio.sleep(0)
        except: pass
    await asyncio.gather(rx(), sim())
print("Server ready")

## 6. Start with public URL

In [0]:
# cloudflared - no account needed
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
import subprocess, time as _time
proc = subprocess.Popen(["./cloudflared-linux-amd64", "tunnel", "--url", "http://localhost:8000"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
_time.sleep(8)
for line in proc.stderr:
    l = line.decode()
    if "trycloudflare.com" in l:
        url = [w for w in l.split() if "trycloudflare.com" in w][0]
        print(f"\nPublic URL: {url}")
        print("Open this in your browser!")
        break

In [0]:
# Start server
import nest_asyncio
nest_asyncio.apply()
print("Starting brain on GPU...")
uvicorn.run(app, host="0.0.0.0", port=8000)